In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import fiona


In [2]:
gdb_file =f'../../../data/Typology_and_Restoration_Potential/Data/Typology_and_Restoration_Potential/Data/MOW_Global_Mangrove_Restoration_20190411.gdb'
layers = fiona.listlayers(gdb_file)

In [3]:
layers

['Global_Mangrove_Restoration_Jurisdiction_Envelope',
 'Global_Mangrove_Restoration_Typology']

In [4]:
layer0 = gpd.read_file(gdb_file, driver='FileGDB', layer=0)

In [5]:
layer0.head()

,Country,Tot_Restor,Restor_pct,Rest_Area_Loss,Area_loss_pct,Rest_Area_Dgrd,Area_dgrd_pct,Total_2016,Mean_Score,SOC,AGB,People,Fish_Score,Fish_Score_Inv,Shape_Length,Shape_Area,Loss_Driver,geometry
0,American Samoa,0.00000,0,0.00,0,0.00,0,18.74,0,0.000000,0.000000,0,109000000.0,200000.0,5.350212e+04,1.638363e+07,No Data,"MULTIPOLYGON (((-19011686.866 -1613074.241, -1..."
1,Angola,580.11556,4,671.43,5,68.92,0,13286.05,62,220274.238512,30809.263129,3800,-1.0,-1.0,1.383794e+06,2.785231e+10,NPC,"MULTIPOLYGON (((1550210.491 -1230695.594, 1549..."
2,Anguilla,0.08128,4,1.27,59,0.00,0,0.87,24,407.670000,6.524752,0,1000000.0,2000000.0,2.097797e+04,1.419573e+06,No Data,"MULTIPOLYGON (((-7025249.376 2058315.576, -702..."
3,Antigua and Barbuda,14.15571,2,20.19,2,0.00,0,886.30,22,9031.556968,1078.042000,500,8000000.0,13000000.0,1.888816e+05,1.361710e+09,No Data,"MULTIPOLYGON (((-6873186.951 1921601.093, -688..."
4,Aruba,19.38207,36,20.55,38,0.00,0,33.79,46,9816.581150,1374.874165,0,71000000.0,119000000.0,3.798229e+04,2.166772e+07,No Data,"MULTIPOLYGON (((-7781925.061 1393950.107, -778..."


### Data Models:

**Restoration potential:**  
{  
location_id [str]  
indicator [str] - restorable_area, mangrove_area, restoration_potential_score  
value [numeric]  
unit [str] - ha, %  
}  

**Degradation:**  
{  
location_id [str]  
indicator [str] - degraded_area, lost_area, main_loss_driver  
value [numeric]  
unit [str] - ha, %  
}  



### Fields:
restorable_area = Tot_Restor  
mangrove_area  = Total_2016  
restoration_potential_score  =  Mean_Score  

degraded_area = Rest_Area_Dgrd  
lost_area = Area_loss_ha  
main_loss_driver = ??

In [6]:
locations = pd.read_csv('../../../data/staging_locations.csv')
locations = locations[locations['location_type'] == 'country']
locations = locations[['id', 'name', 'iso']].copy()
locations.head()

,id,name,iso
159,1402,Dominican Republic,DOM
160,1401,Colombia,COL
161,1400,"Congo, DRC",COD
162,1399,Australia,AUS
163,1398,Angola,AGO


In [7]:
len(locations)

102

In [103]:
table = layer0[['Country']]
table.head()

,Country
0,American Samoa
1,Angola
2,Anguilla
3,Antigua and Barbuda
4,Aruba


In [9]:
len(table['Country'].unique())

108

In [10]:
gadm = gpd.read_file('/Users/sofia/Documents/HE_Data/NRC/NRC_Terrestrial/gadm36_level0_original/gadm36_level0_original.shp')
gadm.head()

,GID_0,NAME_0,AREA_KM2,MOL_ID,Shape_Leng,Shape_Area,geometry
0,ABW,Aruba,1.819384e+02,1,0.963634,0.015131,"POLYGON ((-69.97820 12.46986, -69.97847 12.469..."
1,AFG,Afghanistan,6.438575e+05,2,57.103371,62.749594,"POLYGON ((68.52644 31.75435, 68.53852 31.75457..."
2,AGO,Angola,1.247422e+06,3,73.796528,103.818655,"MULTIPOLYGON (((11.73347 -16.67255, 11.73347 -..."
3,AIA,Anguilla,8.330331e+01,4,1.318321,0.007116,"MULTIPOLYGON (((-63.42375 18.58903, -63.42375 ..."
4,ALA,Åland,1.506261e+03,5,42.232199,0.243769,"MULTIPOLYGON (((21.32195 59.74986, 21.32195 59..."


In [11]:
isocodes = gadm[['GID_0', 'NAME_0']]

In [104]:
table = pd.merge(table, isocodes, how='left', left_on='Country', right_on='NAME_0')
table.head()

,Country,GID_0,NAME_0
0,American Samoa,ASM,American Samoa
1,Angola,AGO,Angola
2,Anguilla,AIA,Anguilla
3,Antigua and Barbuda,ATG,Antigua and Barbuda
4,Aruba,ABW,Aruba


In [96]:
table[table['GID_0'].isnull()]

,Country,GID_0,NAME_0
12,Bonaire,NaN,NaN
30,East Timor,NaN,NaN
83,Sao Tome and Principe,NaN,NaN


In [105]:
table.GID_0[12] = 'BES'
table.GID_0[30] = 'TLS' # East Timor is now called Timor-Leste and the iso is TLS
table.GID_0[83] = 'STP'

In [27]:
table[table['GID_0'].isnull()]

,Country,GID_0,NAME_0


In [106]:
table = table.drop(columns='NAME_0').rename(columns={'GID_0':'iso'})
table.head()

,Country,iso
0,American Samoa,ASM
1,Angola,AGO
2,Anguilla,AIA
3,Antigua and Barbuda,ATG
4,Aruba,ABW


In [107]:
table = pd.merge(table, locations, how='left', on='iso').drop(columns={'name'})
table

,Country,iso,id
0,American Samoa,ASM,NaN
1,Angola,AGO,1398.0
2,Anguilla,AIA,NaN
3,Antigua and Barbuda,ATG,1370.0
4,Aruba,ABW,NaN
...,...,...,...
103,Vanuatu,VUT,1394.0
104,Venezuela,VEN,1362.0
105,Vietnam,VNM,1364.0
106,"Virgin Islands, U.S.",VIR,1397.0


In [108]:
ind1 = table.copy()
ind1['indicator'] = 'restorable_area'
ind1.head()

,Country,iso,id,indicator
0,American Samoa,ASM,NaN,restorable_area
1,Angola,AGO,1398.0,restorable_area
2,Anguilla,AIA,NaN,restorable_area
3,Antigua and Barbuda,ATG,1370.0,restorable_area
4,Aruba,ABW,NaN,restorable_area


In [109]:
ind1 = pd.merge(ind1,layer0[['Country','Tot_Restor']],on='Country', how='left').rename(columns={'Tot_Restor':'value'})
ind1['unit']='ha'
ind1.head()

,Country,iso,id,indicator,value,unit
0,American Samoa,ASM,NaN,restorable_area,0.00000,ha
1,Angola,AGO,1398.0,restorable_area,580.11556,ha
2,Anguilla,AIA,NaN,restorable_area,0.08128,ha
3,Antigua and Barbuda,ATG,1370.0,restorable_area,14.15571,ha
4,Aruba,ABW,NaN,restorable_area,19.38207,ha


restorable_area = Tot_Restor  
mangrove_area  = Total_2016  
restoration_potential_score  =  Mean_Score  

In [110]:
ind2 = table.copy()
ind2['indicator'] = 'mangrove_area'
ind2.head()

,Country,iso,id,indicator
0,American Samoa,ASM,NaN,mangrove_area
1,Angola,AGO,1398.0,mangrove_area
2,Anguilla,AIA,NaN,mangrove_area
3,Antigua and Barbuda,ATG,1370.0,mangrove_area
4,Aruba,ABW,NaN,mangrove_area


In [111]:
ind2 = pd.merge(ind2,layer0[['Country','Total_2016']],on='Country', how='left').rename(columns={'Total_2016':'value'})
ind2['unit']='ha'
ind2.head()

,Country,iso,id,indicator,value,unit
0,American Samoa,ASM,NaN,mangrove_area,18.74,ha
1,Angola,AGO,1398.0,mangrove_area,13286.05,ha
2,Anguilla,AIA,NaN,mangrove_area,0.87,ha
3,Antigua and Barbuda,ATG,1370.0,mangrove_area,886.30,ha
4,Aruba,ABW,NaN,mangrove_area,33.79,ha


In [113]:
ind3 = table.copy()
ind3['indicator'] = 'restoration_potential_score'
ind3.head()

,Country,iso,id,indicator
0,American Samoa,ASM,NaN,restoration_potential_score
1,Angola,AGO,1398.0,restoration_potential_score
2,Anguilla,AIA,NaN,restoration_potential_score
3,Antigua and Barbuda,ATG,1370.0,restoration_potential_score
4,Aruba,ABW,NaN,restoration_potential_score


In [114]:
ind3 = pd.merge(ind3,layer0[['Country','Mean_Score']],on='Country', how='left').rename(columns={'Mean_Score':'value'})
ind3['unit']=''
ind3.head()

,Country,iso,id,indicator,value,unit
0,American Samoa,ASM,NaN,restoration_potential_score,0,
1,Angola,AGO,1398.0,restoration_potential_score,62,
2,Anguilla,AIA,NaN,restoration_potential_score,24,
3,Antigua and Barbuda,ATG,1370.0,restoration_potential_score,22,
4,Aruba,ABW,NaN,restoration_potential_score,46,


In [115]:
table2 = pd.concat([ind1, ind2, ind3], ignore_index=True)
table2 


,Country,iso,id,indicator,value,unit
0,American Samoa,ASM,NaN,restorable_area,0.00000,ha
1,Angola,AGO,1398.0,restorable_area,580.11556,ha
2,Anguilla,AIA,NaN,restorable_area,0.08128,ha
3,Antigua and Barbuda,ATG,1370.0,restorable_area,14.15571,ha
4,Aruba,ABW,NaN,restorable_area,19.38207,ha
...,...,...,...,...,...,...
319,Vanuatu,VUT,1394.0,restoration_potential_score,64.00000,
320,Venezuela,VEN,1362.0,restoration_potential_score,57.00000,
321,Vietnam,VNM,1364.0,restoration_potential_score,54.00000,
322,"Virgin Islands, U.S.",VIR,1397.0,restoration_potential_score,29.00000,


In [117]:
table2.to_csv('../../../data/RestorationPotential.csv')